# Week 3 · Day 1: Create a classification target

**Course:** Lab 01: Classification

The teaching question is **“Is this source's absolute G magnitude below 4?”** Class 1 means yes. This is a brightness label, not a verified stellar type. Only `bp_rp` (color) enters the model; magnitude and parallax are used to construct the answer and must stay out of the inputs.

Day 1 saves a source-disjoint 60/20/20 split. Days 2–6 explore training and validation; Day 7 evaluates the reserved test set. Absolute magnitudes are uncorrected for dust, and the archive sample may not represent all Gaia sources.

## Setup

[Prerequisites and setup explanation](week3_prerequisites.ipynb). Run cells from top to bottom. This day creates the files used by the following days.

In [ ]:
from pathlib import Path
import sys
import importlib
import numpy as np
import pandas as pd
from IPython.display import display

# Jupyter may start in the project folder or in its notebooks folder.
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'src/classification.py').is_file():
    raise RuntimeError('Open Jupyter from gaia-explorer or gaia-explorer/notebooks.')
sys.path.insert(0, str(ROOT))
from src import classification as data, visualize
# Refresh the data loader if this kernel imported an older version.
data = importlib.reload(data)
visualize = importlib.reload(visualize)  # Pick up updated plot labels and layout.

DATA_PATH = None  # None uses the shared W2 cache; a missing cache is downloaded.
OUTPUT = ROOT / 'data/processed/week3'
OUTPUT.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 20)


## 1. Load the measurements

Reuse the W2 cache. If it is missing, the shared loader downloads real Gaia data. It removes missing or invalid measurements, keeps positive parallax with SNR > 5, and removes duplicate source IDs. Expected output: row counts.

In [ ]:
gaia, audit = data.load_week3_data(DATA_PATH)
display(audit)
display(gaia.head())

## 2. Calculate absolute magnitude

This is the W2 calculation, now visible in the notebook. Parallax is in milliarcseconds. Smaller magnitude means brighter. Expected output: distances and magnitudes.

In [ ]:
gaia['distance_pc'] = 1000 / gaia['parallax']
gaia['abs_g_mag'] = gaia['phot_g_mean_mag'] + 5 * np.log10(gaia['parallax']) - 10
display(gaia[['source_id', 'distance_pc', 'abs_g_mag']].head())

## 3. Make the yes/no label

Use 4 as a fixed teaching cut. A magnitude exactly equal to 4 belongs to class 0. This cut does not determine whether a star is a giant. Expected output: a binary column.

In [ ]:
magnitude_cut = 4.0
gaia['is_bright'] = (gaia['abs_g_mag'] < magnitude_cut).astype(int)
gaia['distance_to_label_cut_mag'] = abs(gaia['abs_g_mag'] - magnitude_cut)
print('Class counts:', gaia['is_bright'].value_counts().to_dict())
display(gaia[['bp_rp', 'abs_g_mag', 'is_bright']].head(10))

## 4. Split before learning

Save 60% for training, 20% for validation choices, and 20% for the final test. `stratify` keeps both classes represented. The seed makes the split repeatable.

In [ ]:
from sklearn.model_selection import train_test_split

counts = gaia['is_bright'].value_counts()
if len(counts) != 2 or counts.min() < 10:
    raise ValueError('Use a larger Gaia sample with at least 10 sources in each brightness class.')
train, remaining = train_test_split(gaia, test_size=0.4, stratify=gaia['is_bright'], random_state=42)
validation, test = train_test_split(remaining, test_size=0.5, stratify=remaining['is_bright'], random_state=42)
print('Train / validation / test:', len(train), len(validation), len(test))

## What are X and y?

In machine learning, `X` means the input information the model sees, and `y` means the answer it learns to predict. Each row represents one star. Here, `X_train` has just the star's color (`bp_rp`), while `y_train` has its brightness label (`is_bright`). We leave absolute magnitude out of `X_train` because it was used to create `y_train`.

The later notebooks use these same column choices with the saved training split.

In [ ]:
feature_columns = ['bp_rp']
X_train = train[feature_columns]   # One input column: star color.
y_train = train['is_bright']    # The 0/1 answer.

print('X shape (stars, input columns):', X_train.shape)
print('y shape (stars):', y_train.shape)
display(X_train.head().assign(is_bright=y_train.head()))

## 5. Inspect the training sample

Expected outputs: HR diagram and examples near the label cut. The plot color bar says **blue = 0 (fainter, M_G ≥ 4)** and **red = 1 (brighter, M_G < 4)**. Dot colors show labels; the star’s measured color is its horizontal BP-RP position. Sources close to 4 may change labels with measurement error. A 0.2 mag band is a diagnostic choice, not an uncertainty estimate.

In [ ]:
visualize.plot_classification_hr(train)
near_cut = train.loc[train['distance_to_label_cut_mag'] < 0.2]
display(near_cut[['source_id', 'abs_g_mag', 'parallax_snr', 'is_bright']].head(10))

## 6. Save for Days 2–7

Later notebooks reuse these exact rows and splits. If you experiment with another magnitude cut, restore 4 and rerun Day 1 before continuing. Rerunning replaces the W3 prepared data; rerun later days after changing it.

In [ ]:
prepared = pd.concat([
    train.assign(split='train'),
    validation.assign(split='validation'),
    test.assign(split='test'),
], ignore_index=True)
prepared.to_csv(OUTPUT / 'learning_data.csv', index=False)
prepared[['source_id', 'split']].to_csv(OUTPUT / 'split_manifest.csv', index=False)
print('Saved:', OUTPUT / 'learning_data.csv')

## Reflection questions and answers

1. **What changed from W2?** In W2, we predicted a number: the star's absolute G magnitude (`abs_g_mag`). For example, the answer might be `3.5` or `6.2`. In W3, we turn that number into one of two labels by asking whether it is below 4. Predicting a number is **regression**; predicting a label is **classification**.

2. **Why exclude magnitude from X?** `X` contains the information the model is allowed to use to make a prediction. The answer (`is_bright`) is calculated directly from `abs_g_mag`, so giving `abs_g_mag` to the model would give away the answer. We also exclude `phot_g_mean_mag` and `parallax`, because those two measurements are used to calculate `abs_g_mag`. The W3 model uses only `bp_rp` (color) as its input.

3. **What does class 1 mean?** `is_bright = 1` means the calculated absolute G magnitude is **below 4**. `is_bright = 0` means it is **4 or above**. A smaller magnitude number means a brighter star. This cutoff is a choice for our exercise; class 1 does not mean the star is a red giant or any other confirmed stellar type.


## References

Course labs: `../ML-coursework/W3/Optional Labs/` in the workspace.

[ESA Gaia DR3](https://www.cosmos.esa.int/web/gaia/dr3) · [Extinction guidance](https://www.cosmos.esa.int/web/gaia/edr3-extinction-law) · [Scikit-learn logistic regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)